# 机器学习温压计标准化评估协议

本 Notebook 实现 4 组实验的一键运行：
- **Exp1**: CatBoost 基线（无增强、无校正）
- **Exp2**: CatBoost + 数据增强
- **Exp3**: CatBoost + 数据增强 + 偏差校正
- **Exp4**: Stacking + 数据增强 + 偏差校正

**核心协议约束**：
1. 外层评估使用 GroupKFold（按 Ref 分组）
2. T 与 P 采用独立建模链路
3. 所有拟合操作只在训练折内完成
4. 每折结束立即落盘

## 1. 环境初始化

In [ ]:
# 导入系统模块
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 设置项目根目录
PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"项目根目录: {PROJECT_ROOT}")

In [ ]:
# 导入框架模块
from src import (
    # 模型
    get_model, CatBoostWrapper, GroupAwareStacker, create_default_stacker,
    # 运行器
    ExperimentConfig, ExperimentRunner, run_experiment_matrix,
    # 预处理
    load_data, prepare_data, get_feature_cols,
    # 指标
    summarize_folds, print_summary, compare_experiments,
    # 可视化
    plot_pred_vs_true, plot_residuals, plot_full_report, plot_experiment_summary
)
from config import (
    DATA_PATH, OUTPUT_DIR, RANDOM_SEED, N_SPLITS,
    CATBOOST_DEFAULT_PARAMS, EXPERIMENT_CONFIGS
)

print("✅ 所有模块导入成功！")

## 2. 数据加载与预处理

In [ ]:
# 加载数据
data_path = os.path.join(PROJECT_ROOT, 'input.csv')
df = load_data(data_path, encoding='latin-1')

print(f"数据形状: {df.shape}")
print(f"\n前5行预览:")
df.head()

In [ ]:
# 准备实验数据
data = prepare_data(df, feature_mode='cpx_liq')

print(f"特征矩阵形状: {data['X'].shape}")
print(f"特征列数: {len(data['feature_cols'])}")
print(f"\n温度 T 范围: {data['y_T'].min():.0f} - {data['y_T'].max():.0f} ℃")
print(f"压力 P 范围: {data['y_P'].min():.2f} - {data['y_P'].max():.2f} kbar")
print(f"\n文献来源数量: {len(np.unique(data['groups']))}")

## 3. 定义实验配置

In [ ]:
# 实验矩阵配置
configs = [
    # Exp1: CatBoost 基线
    ExperimentConfig(
        exp_name='exp1_catboost_base',
        model_type='catboost',
        model_params={'iterations': 1000, 'depth': 6, 'learning_rate': 0.03, 'silent': True},
        augment=False,
        correct=False,
        n_splits=N_SPLITS,
        output_dir=os.path.join(PROJECT_ROOT, 'outputs')
    ),
    
    # Exp2: CatBoost + 增强
    ExperimentConfig(
        exp_name='exp2_catboost_aug',
        model_type='catboost',
        model_params={'iterations': 1000, 'depth': 6, 'learning_rate': 0.03, 'silent': True},
        augment=True,
        correct=False,
        n_splits=N_SPLITS,
        output_dir=os.path.join(PROJECT_ROOT, 'outputs')
    ),
    
    # Exp3: CatBoost + 增强 + 校正
    ExperimentConfig(
        exp_name='exp3_catboost_aug_corr',
        model_type='catboost',
        model_params={'iterations': 1000, 'depth': 6, 'learning_rate': 0.03, 'silent': True},
        augment=True,
        correct=True,
        n_splits=N_SPLITS,
        output_dir=os.path.join(PROJECT_ROOT, 'outputs')
    ),
    
    # Exp4: Stacking + 增强 + 校正
    ExperimentConfig(
        exp_name='exp4_stacking_aug_corr',
        model_type='stacking',
        model_params={
            'base_models': [
                CatBoostWrapper(iterations=500, depth=4, learning_rate=0.05, silent=True),
                CatBoostWrapper(iterations=800, depth=6, learning_rate=0.03, silent=True),
                CatBoostWrapper(iterations=1000, depth=8, learning_rate=0.02, silent=True),
            ],
            'meta_model': CatBoostWrapper(iterations=300, depth=4, learning_rate=0.05, silent=True),
            'inner_cv': 5,
            'cache_dir': os.path.join(PROJECT_ROOT, 'outputs', 'cache')
        },
        augment=True,
        correct=True,
        n_splits=N_SPLITS,
        output_dir=os.path.join(PROJECT_ROOT, 'outputs')
    ),
]

print(f"共定义 {len(configs)} 个实验配置：")
for cfg in configs:
    print(f"  - {cfg.exp_name}: model={cfg.model_type}, aug={cfg.augment}, corr={cfg.correct}")

## 4. 运行实验矩阵

In [ ]:
# 运行所有实验
all_results = []

for config in configs:
    runner = ExperimentRunner(config)
    results = runner.run_experiment(
        X=data['X'],
        y_T=data['y_T'],
        y_P=data['y_P'],
        groups=data['groups'],
        row_ids=data['row_ids'],
        refs=data['refs']
    )
    all_results.append(results)

## 5. 结果汇总与对比

In [ ]:
# 汇总所有实验结果
results_df = pd.DataFrame(all_results)

# 选择关键指标列
key_cols = ['exp_name', 'rmse_T_mean', 'rmse_T_std', 'r2_T_mean', 'rmse_P_mean', 'rmse_P_std', 'r2_P_mean']
display_cols = [c for c in key_cols if c in results_df.columns]

print("\n" + "="*80)
print("实验结果汇总")
print("="*80)
results_df[display_cols]

In [ ]:
# 可视化对比
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 温度 RMSE 对比
ax1 = axes[0]
x = range(len(results_df))
ax1.bar(x, results_df['rmse_T_mean'], yerr=results_df['rmse_T_std'], capsize=5, alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(results_df['exp_name'], rotation=45, ha='right')
ax1.set_ylabel('RMSE (℃)')
ax1.set_title('温度 T - RMSE 对比')
ax1.grid(axis='y', alpha=0.3)

# 压力 RMSE 对比
ax2 = axes[1]
ax2.bar(x, results_df['rmse_P_mean'], yerr=results_df['rmse_P_std'], capsize=5, alpha=0.8, color='orange')
ax2.set_xticks(x)
ax2.set_xticklabels(results_df['exp_name'], rotation=45, ha='right')
ax2.set_ylabel('RMSE (kbar)')
ax2.set_title('压力 P - RMSE 对比')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', 'experiment_comparison.png'), dpi=150)
plt.show()

## 6. 查看单个实验的详细预测结果

In [ ]:
# 加载最佳实验的预测结果
best_exp = 'exp4_stacking_aug_corr'  # 可修改为其他实验
preds_path = os.path.join(PROJECT_ROOT, 'outputs', best_exp, 'preds.parquet')

if os.path.exists(preds_path):
    preds_df = pd.read_parquet(preds_path)
    print(f"预测结果形状: {preds_df.shape}")
    print(f"\n前10行:")
    display(preds_df.head(10))
else:
    print(f"文件不存在: {preds_path}")

In [ ]:
# 绘制最佳实验的预测-真实散点图
if 'preds_df' in dir():
    fig = plot_full_report(
        y_T_true=preds_df['T_true'].values,
        y_T_pred=preds_df['T_pred_corr'].values,
        y_P_true=preds_df['P_true'].values,
        y_P_pred=preds_df['P_pred_corr'].values,
        exp_name=best_exp
    )
    plt.savefig(os.path.join(PROJECT_ROOT, 'outputs', f'{best_exp}_report.png'), dpi=150)
    plt.show()

## 7. 保存汇总结果

In [ ]:
# 保存实验对比表
summary_path = os.path.join(PROJECT_ROOT, 'outputs', 'all_experiments_summary.csv')
results_df.to_csv(summary_path, index=False)
print(f"汇总结果已保存: {summary_path}")

---
## 完成！

所有实验已运行完成，输出文件位于 `outputs/` 目录：
- `{exp_name}/metrics.csv`: 各折指标
- `{exp_name}/preds.parquet`: 逐样本预测
- `{exp_name}/summary.csv`: 实验汇总
- `all_experiments_summary.csv`: 全部实验对比